In [13]:
!pip install flask requests joblib -q

In [3]:
import pandas as pd

# Paste your GitHub RAW CSV URL here
DATASET_URL = "https://raw.githubusercontent.com/subathaks/Machine-Learning-Lab/refs/heads/main/data.csv"

data = pd.read_csv(DATASET_URL)

print("Dataset loaded successfully!")
print("Shape:", data.shape)
print(data.head())
print("\nColumns:")
print(data.columns)

Dataset loaded successfully!
Shape: (569, 33)
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_

In [4]:
# Remove unnecessary columns
data = data.drop(["id", "Unnamed: 32"], axis=1)

# Convert diagnosis into numbers
data["diagnosis"] = data["diagnosis"].map({
    "M": 1,
    "B": 0
})

# Features and target
X = data.drop("diagnosis", axis=1)
y = data["diagnosis"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (569, 30)
Target shape: (569,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Model created successfully!")

Model created successfully!


In [7]:
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [8]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9736842105263158

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98        71
           1       0.98      0.95      0.96        43

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [9]:
import joblib

joblib.dump(model, "model.pkl")

print("Model trained and saved successfully!")

Model trained and saved successfully!


In [10]:
%%writefile app.py

from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)

# Load trained ML model
model = joblib.load("model.pkl")


@app.route("/")
def home():
    return "ML Model API is running!"


@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    features = data["features"]

    prediction = model.predict([features])[0]

    if prediction == 1:
        result = "Malignant"
    else:
        result = "Benign"

    return jsonify({
        "prediction": int(prediction),
        "result": result
    })


if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5001,
        debug=False
    )

Overwriting app.py


In [11]:
import threading
import time
from app import app

def run_flask():
    app.run(
        host="127.0.0.1",
        port=5001,
        debug=False,
        use_reloader=False
    )

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

time.sleep(3)

print("Flask server started successfully on port 5001!")

 * Serving Flask app 'app'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


Flask server started successfully on port 5001!


In [12]:
import requests

data = {
    "features": [
        17.99, 10.38, 122.8, 1001.0, 0.1184,
        0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
        1.095, 0.9053, 8.589, 153.4, 0.006399,
        0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
        25.38, 17.33, 184.6, 2019.0, 0.1622,
        0.6656, 0.7119, 0.2654, 0.4601, 0.1189
    ]
}

response = requests.post(
    "http://127.0.0.1:5001/predict",
    json=data
)

print("Status Code:", response.status_code)
print("API Response:")
print(response.json())

C:\Users\Ajith Prasad Shetty\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [05/Sep/2026 20:55:43] "POST /predict HTTP/1.1" 200 -


Status Code: 200
API Response:
{'prediction': 1, 'result': 'Malignant'}
